In [29]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

In [30]:
# load back in the RNA fingerprints and split prepared in Task3_01
DATA_DIR = "/home/ubuntu/data/frangieh"

pert_FC_selected = pd.read_pickle(f"{DATA_DIR}/task3_pert_FC_selected_50.pkl")
train_40 = pd.read_csv(f"{DATA_DIR}/task3_train_40.csv")["perturbation"].tolist()
test_10 = pd.read_csv(f"{DATA_DIR}/task3_test_10.csv")["perturbation"].tolist()
gene_cols = pert_FC_selected.columns.tolist()
selected_50 = train_40 + test_10
conditions = pert_FC_selected.index.get_level_values("condition").unique().tolist()

pert_FC_selected.shape

(150, 2042)

In [6]:
pert_FC_selected

gene_symbol               A2M-AS1     AADAC     AATBC    ABCB11     ABCB6  \
perturbation condition                                                      
BOLA2B       Control    -0.000482  0.004264 -0.001478 -0.000222  0.002504   
             IFNγ       -0.000703 -0.005169 -0.000362 -0.003143 -0.004518   
             Co-culture -0.000265 -0.001347 -0.000155 -0.000559  0.007231   
STOM         Control    -0.000482  0.006261 -0.001478 -0.000222 -0.007542   
             IFNγ       -0.000703 -0.005169 -0.000362  0.005771 -0.007454   
...                           ...       ...       ...       ...       ...   
NEAT1        IFNγ       -0.000703 -0.005169 -0.000362  0.008721 -0.003324   
             Co-culture -0.000265 -0.001347 -0.000155  0.009161 -0.003508   
S100B        Control    -0.000482 -0.005252 -0.001478 -0.000222  0.035455   
             IFNγ       -0.000703 -0.005169 -0.000362 -0.003143  0.000763   
             Co-culture -0.000265 -0.001347 -0.000155 -0.000559  0.008512   

gene_symbol               ABHD12B      ABL2  AC002310.2  AC002347.1  \
perturbation condition                                                
BOLA2B       Control    -0.000309  0.039339    0.003140   -0.002348   
             IFNγ       -0.000694  0.031402   -0.003718   -0.003534   
             Co-culture  0.000000 -0.103634    0.003129   -0.000671   
STOM         Control    -0.000309 -0.034413   -0.005888    0.000943   
             IFNγ       -0.000694  0.010152   -0.001878    0.005380   
...                           ...       ...         ...         ...   
NEAT1        IFNγ       -0.000694 -0.011371    0.005114   -0.003534   
             Co-culture  0.000000  0.032517    0.003226    0.002409   
S100B        Control    -0.000309  0.061824   -0.005888   -0.002348   
             IFNγ       -0.000694  0.104629   -0.000594   -0.003534   
             Co-culture  0.000000  0.016085   -0.001820   -0.000671   

gene_symbol              AC002384.1  ...    ZNF366    ZNF404  ZNF451-AS1  \
perturbation condition               ...                                   
BOLA2B       Control       0.005321  ... -0.000818 -0.004023   -0.001394   
             IFNγ         -0.000560  ... -0.000257  0.003989    0.012441   
             Co-culture   -0.000118  ... -0.000137  0.008035   -0.003728   
STOM         Control      -0.001842  ... -0.000818 -0.007187   -0.010138   
             IFNγ         -0.000560  ... -0.000257  0.001207    0.002538   
...                             ...  ...       ...       ...         ...   
NEAT1        IFNγ         -0.000560  ... -0.000257  0.014535   -0.007310   
             Co-culture   -0.000118  ... -0.000137 -0.004089    0.006244   
S100B        Control       0.000975  ... -0.000818 -0.003778   -0.010138   
             IFNγ         -0.000560  ... -0.000257  0.007208    0.000407   
             Co-culture   -0.000118  ... -0.000137  0.007651    0.007909   

gene_symbol                ZNF534    ZNF556    ZNF660    ZNF728    ZNF763  \
perturbation condition                                                      
BOLA2B       Control    -0.002173 -0.002489 -0.000669 -0.010135 -0.003664   
             IFNγ       -0.003263 -0.001641 -0.002699 -0.008548 -0.004467   
             Co-culture -0.001019 -0.001941  0.000000 -0.002331 -0.001209   
STOM         Control     0.001832 -0.002489 -0.000669 -0.005802 -0.003664   
             IFNγ       -0.003263 -0.001641 -0.002699 -0.002444 -0.004467   
...                           ...       ...       ...       ...       ...   
NEAT1        IFNγ        0.016814 -0.001641 -0.002699 -0.008548 -0.004467   
             Co-culture -0.001019 -0.001941  0.000000 -0.002331  0.016030   
S100B        Control    -0.002173  0.001076 -0.000669 -0.004708 -0.003664   
             IFNγ        0.004600  0.002795 -0.002699  0.001818 -0.004467   
             Co-culture -0.001019 -0.001941  0.000000 -0.002331  0.002391   

gene_symbol                ZNF844   ZSCAN5B  
perturbation condition               

In [24]:
pert_FC_selected.xs("Control",level="condition")

gene_symbol,A2M-AS1,AADAC,AATBC,ABCB11,ABCB6,ABHD12B,ABL2,AC002310.2,AC002347.1,AC002384.1,...,ZNF366,ZNF404,ZNF451-AS1,ZNF534,ZNF556,ZNF660,ZNF728,ZNF763,ZNF844,ZSCAN5B
perturbation,,,,,,,,,,,,,,,,,,,,,
BOLA2B,-0.000482,0.004264,-0.001478,-0.000222,0.002504,-0.000309,0.039339,0.003140,-0.002348,0.005321,...,-0.000818,-0.004023,-0.001394,-0.002173,-0.002489,-0.000669,-0.010135,-0.003664,-0.002285,-0.001069
STOM,-0.000482,0.006261,-0.001478,-0.000222,-0.007542,-0.000309,-0.034413,-0.005888,0.000943,-0.001842,...,-0.000818,-0.007187,-0.010138,0.001832,-0.002489,-0.000669,-0.005802,-0.003664,0.012656,0.050588
LY96,-0.000482,0.011010,-0.001478,0.002066,0.001595,-0.000309,0.057202,-0.005888,-0.002348,-0.001842,...,-0.000818,-0.003431,-0.003059,0.004152,0.000951,0.004346,-0.010135,-0.003664,0.007889,-0.001069
SCARB2,-0.000482,-0.002948,0.004681,-0.000222,0.009565,-0.000309,-0.145667,-0.005888,-0.002348,-0.001842,...,-0.000818,-0.007187,0.004546,-0.002173,0.007276,-0.000669,0.005226,0.001143,-0.013331,-0.001069
HASPIN,-0.000482,-0.005252,-0.001478,-0.000222,-0.007542,-0.000309,-0.122139,-0.005888,-0.002348,-0.001842,...,-0.000818,-0.002193,0.005093,0.008936,-0.002489,-0.000669,-0.004012,-0.003664,-0.018569,-0.001069
CCR10,-0.000482,0.009540,0.004483,-0.000222,0.003665,-0.000309,-0.014933,0.003404,-0.000275,-0.001842,...,-0.000818,-0.007187,0.002750,-0.002173,-0.002489,0.010755,-0.004585,0.000760,-0.000835,-0.001069
AHNAK,-0.000482,-0.001354,-0.001478,-0.000222,-0.004304,-0.000309,0.077827,-0.004124,-0.002348,-0.001842,...,-0.000818,0.002847,0.000568,-0.000130,-0.002489,-0.000669,-0.007187,-0.003664,0.002454,-0.001069
TTLL1,-0.000482,0.012980,-0.001478,-0.000222,-0.007542,-0.000309,0.083486,-0.005888,-0.002348,-0.001842,...,-0.000818,-0.000178,-0.005164,-0.002173,0.009509,-0.000669,-0.010135,-0.003664,0.020225,-0.001069
S100A6,-0.000482,-0.005252,-0.001478,-0.000222,0.004183,-0.000309,-0.035564,-0.005888,-0.002348,-0.001842,...,-0.000818,-0.002277,-0.001500,-0.002173,-0.002489,-0.000669,-0.010135,0.000996,-0.000404,-0.001069


In [35]:
# ---------------------------------------------------------------
# 2. Gene "identity" features (leakage-free, generalizes to unseen genes)
# ---------------------------------------------------------------
def build_gene_embeddings(fc_df, train_perts, gene_cols, n_components=32):
    """
    Embed every gene in `gene_cols` using its own column of log2FC values,
    restricted to rows whose perturbation is in `train_perts`.
    Shape going in: (n_train_rows, n_genes) -> transpose -> genes as samples.
    """
    train_rows = fc_df.index.get_level_values("perturbation").isin(train_perts)
    train_block = fc_df.loc[train_rows]  # (n_train_rows, n_genes)

    gene_signatures = train_block.T.values  # (n_genes, n_train_rows)

    scaler = StandardScaler()
    gene_signatures = scaler.fit_transform(gene_signatures)

    n_components = min(n_components, gene_signatures.shape[0], gene_signatures.shape[1])
    pca = PCA(n_components=n_components, random_state=SEED)
    embeddings = pca.fit_transform(gene_signatures)  # (n_genes, n_components)

    emb_df = pd.DataFrame(embeddings, index=gene_cols)
    return emb_df, pca, scaler


def get_gene_embedding(gene, emb_df, dim):
    if gene in emb_df.index:
        return emb_df.loc[gene].values.astype(np.float32)
    # gene wasn't itself profiled as a measured column -> fall back to zeros
    return np.zeros(dim, dtype=np.float32)


gene_embeddings, gene_pca, gene_scaler = build_gene_embeddings(
    pert_FC_selected, train_40, gene_cols, n_components=32
)
EMB_DIM = gene_embeddings.shape[1]

# ---------------------------------------------------------------
# 3. Target compression (PCA fit on train targets only)
# ---------------------------------------------------------------
train_mask = pert_FC_selected.index.get_level_values("perturbation").isin(train_40)
Y_train_full = pert_FC_selected.loc[train_mask].values

N_TARGET_PCS = min(35, Y_train_full.shape[0] - 1)  # keep well below n_train_rows
target_pca = PCA(n_components=N_TARGET_PCS, random_state=SEED)
target_pca.fit(Y_train_full)
print(f"Target PCA explained variance ({N_TARGET_PCS} PCs): "
      f"{target_pca.explained_variance_ratio_.sum():.3f}")

# ---------------------------------------------------------------
# 4. Build (X, Y) arrays, one row per (perturbation, condition)
# ---------------------------------------------------------------
cond_to_idx = {c: i for i, c in enumerate(conditions)}

def build_dataset(perturbations, fc_df, emb_df, target_pca_model):
    X, Y_pc, Y_full, meta = [], [], [], []
    for pert in perturbations:
        sub = fc_df.loc[pert]  # index = condition
        for cond in sub.index.get_level_values("condition"):
            fc_vec = sub.loc[cond].values.astype(np.float32)
            gene_emb = get_gene_embedding(pert, emb_df, EMB_DIM)
            cond_oh = np.zeros(len(conditions), dtype=np.float32)
            cond_oh[cond_to_idx[cond]] = 1.0
            X.append(np.concatenate([gene_emb, cond_oh]))
            Y_pc.append(target_pca_model.transform(fc_vec[None, :])[0])
            Y_full.append(fc_vec)
            meta.append((pert, cond))
    return (np.stack(X).astype(np.float32),
            np.stack(Y_pc).astype(np.float32),
            np.stack(Y_full).astype(np.float32),
            meta)

# hold out a validation split from train_40, split by PERTURBATION (not row)
# so validation genes are also "unseen" during training -- mirrors test_10.
val_perts = list(np.random.choice(train_40, size=max(1, len(train_40) // 5), replace=False))
fit_perts = [p for p in train_40 if p not in val_perts]

X_fit, Y_fit_pc, _, meta_fit = build_dataset(fit_perts, pert_FC_selected, gene_embeddings, target_pca)
X_val, Y_val_pc, Y_val_true, meta_val = build_dataset(val_perts, pert_FC_selected, gene_embeddings, target_pca)
X_test, Y_test_pc, Y_test_true, meta_test = build_dataset(test_10, pert_FC_selected, gene_embeddings, target_pca)

# ---------------------------------------------------------------
# 5. Dataset / DataLoader
# ---------------------------------------------------------------
class PertDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X)
        self.Y = torch.from_numpy(Y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.Y[i]


train_loader = DataLoader(PertDataset(X_fit, Y_fit_pc), batch_size=16, shuffle=True)

# ---------------------------------------------------------------
# 6. Model
# ---------------------------------------------------------------
class PerturbationMLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=128, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, x):
        return self.net(x)


model = PerturbationMLP(in_dim=X_fit.shape[1], out_dim=N_TARGET_PCS)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.MSELoss()

# ---------------------------------------------------------------
# 7. Train (early-stopping on held-out-perturbation validation set)
# ---------------------------------------------------------------
N_EPOCHS = 200
best_val = np.inf
best_state = None

X_val_t = torch.from_numpy(X_val)
Y_val_pc_t = torch.from_numpy(Y_val_pc)

for epoch in range(N_EPOCHS):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_t)
        val_loss = loss_fn(val_pred, Y_val_pc_t).item()
    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    if epoch % 20 == 0:
        print(f"epoch {epoch:3d}  val_mse={val_loss:.4f}")

model.load_state_dict(best_state)

# ---------------------------------------------------------------
# 8. Baseline: per-condition mean of training perturbations
# ---------------------------------------------------------------
def condition_mean_baseline(fc_df, train_perts):
    train_rows = fc_df.loc[fc_df.index.get_level_values("perturbation").isin(train_perts)]
    return train_rows.groupby(level="condition").mean()  # (n_conditions, n_genes)

baseline_means = condition_mean_baseline(pert_FC_selected, train_40)

# ---------------------------------------------------------------
# 9. Evaluate on held-out test_10 perturbations
# ---------------------------------------------------------------
def evaluate(model, target_pca_model, X, Y_true_full, meta, baseline_means):
    model.eval()
    with torch.no_grad():
        pred_pc = model(torch.from_numpy(X)).numpy()
    pred_full = target_pca_model.inverse_transform(pred_pc)

    rows = []
    for i, (pert, cond) in enumerate(meta):
        y_true = Y_true_full[i]
        y_pred = pred_full[i]
        y_base = baseline_means.loc[cond].values

        r_model, _ = pearsonr(y_true, y_pred)
        r_base, _ = pearsonr(y_true, y_base)
        mse_model = np.mean((y_true - y_pred) ** 2)
        mse_base = np.mean((y_true - y_base) ** 2)
        spearman_r_model, _ = spearmanr(y_true,y_pred)
        spearman_r_base, _ = spearmanr(y_true,y_base)
        rows.append(dict(perturbation=pert, condition=cond,
                          pearson_model=r_model, pearson_baseline=r_base,
                          mse_model=mse_model, mse_baseline=mse_base, spearman_base = spearman_r_base, spearman_model = spearman_r_model))
    return pd.DataFrame(rows)

results_df = evaluate(model, target_pca, X_test, Y_test_true, meta_test, baseline_means)
print(results_df.groupby("condition")[["pearson_model", "pearson_baseline",
                                        "mse_model", "mse_baseline", "spearman_base", "spearman_model"]].mean())
print(results_df)


Target PCA explained variance (35 PCs): 0.811
epoch   0  val_mse=0.0885
epoch  20  val_mse=0.0842
epoch  40  val_mse=0.0882
epoch  60  val_mse=0.0984
epoch  80  val_mse=0.0974
epoch 100  val_mse=0.1010
epoch 120  val_mse=0.1062
epoch 140  val_mse=0.1123
epoch 160  val_mse=0.1135
epoch 180  val_mse=0.1170
            pearson_model  pearson_baseline  mse_model  mse_baseline  \
condition                                                              
Co-culture       0.565203          0.592829   0.004305      0.003967   
Control          0.752311          0.754654   0.002507      0.002500   
IFNγ             0.711404          0.717705   0.002388      0.002430   

            spearman_base  spearman_model  
condition                                  
Co-culture       0.267087        0.235031  
Control          0.398926        0.339533  
IFNγ             0.379795        0.306897  
   perturbation   condition  pearson_model  pearson_baseline  mse_model  \
0         KCNN4     Control       0.82